# RSNN on FI-2010 — Fixing the Threshold Mismatch

**Diagnosed failure**: LOB inter-timestep changes are O(10⁻²), LIF threshold is 1.0 → neurons don't fire meaningfully.

**This notebook tests three targeted fixes** (all on the V1 balanced test set, 139K samples):

| Experiment | What it changes | Reference |
|---|---|---|
| E1: Learnable threshold | Per-layer trainable V_th initialised near data scale | LTMD (Wang et al., NeurIPS 2022) |
| E2: Batch Norm Through Time | Per-timestep normalisation before LIF | BNTT (Kim & Panda, Frontiers Neurosci 2021) |
| E3: Learned input gain | Per-feature learnable scalar before LIF input | Lightweight BNTT ablation |

**Controls**: Same data split as V1 (days 1-7 train 80/20, days 8-9 test = 139,488 samples). Same Trainer, Adamax optimiser, class-weighted CE loss, spike regularisation, early stopping.

**V1 baselines (carried forward)**:
| Model | Test Acc |
|---|---|
| RSNN Direct (threshold=1.0) | 37.56% |
| LSTM (2L, 128) | 66.07% |
| 1D-CNN | 64.65% |
| Random | 33.3% |


In [1]:
import os, glob, subprocess, time, json, copy
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, f1_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True


Device: cuda
GPU: Tesla T4


## 1. Load FI-2010 Data (V1 protocol)

In [2]:
# Download from DeepLOB repository
print("Downloading FI-2010...")
subprocess.run(['wget', '-q',
    'https://raw.githubusercontent.com/zcakhaa/DeepLOB-Deep-Convolutional-Neural-Networks-for-Limit-Order-Books/master/data/data.zip',
    '-O', '/kaggle/working/data.zip'], check=True)
subprocess.run(['unzip', '-q', '-o', '/kaggle/working/data.zip', '-d', '/kaggle/working/'], check=True)
DATA_DIR = '/kaggle/working'
print("Done.")

train_files = sorted(glob.glob(os.path.join(DATA_DIR, "Train_Dst_NoAuction*.txt")))
test_files = sorted(glob.glob(os.path.join(DATA_DIR, "Test_Dst_NoAuction*.txt")))
print(f"Train files: {[os.path.basename(f) for f in train_files]}")
print(f"Test files: {[os.path.basename(f) for f in test_files]}")


Done.
Train files: ['Train_Dst_NoAuction_DecPre_CF_7.txt']
Test files: ['Test_Dst_NoAuction_DecPre_CF_7.txt', 'Test_Dst_NoAuction_DecPre_CF_8.txt', 'Test_Dst_NoAuction_DecPre_CF_9.txt']


In [3]:
def prepare_x(data):
    return data[:40, :].T.astype(np.float32)

def get_label(data):
    labels = data[-5:, :].T.astype(int) - 1
    return labels

def data_classification(X, Y, T=100):
    N = X.shape[0]
    samples = N - T + 1
    X_seq = np.zeros((samples, T, X.shape[1]), dtype=np.float32)
    for i in range(samples):
        X_seq[i] = X[i:i+T]
    Y_seq = Y[T-1:]
    return X_seq, Y_seq

# Load training data (days 1-7)
dec_train = np.loadtxt(os.path.join(DATA_DIR, 'Train_Dst_NoAuction_DecPre_CF_7.txt'))

# Load ALL 3 test files (V1 protocol — balanced test set)
test_data_list = [np.loadtxt(tf) for tf in test_files]
dec_test = np.hstack(test_data_list)
print(f"Raw train: {dec_train.shape}, Raw test: {dec_test.shape}")

train_lob, train_label = prepare_x(dec_train), get_label(dec_train)
test_lob, test_label = prepare_x(dec_test), get_label(dec_test)

T = 100
HORIZON = 3  # k=50
HORIZON_NAMES = ['k=10', 'k=20', 'k=30', 'k=50', 'k=100']

X_train_seq, y_train_seq = data_classification(train_lob, train_label, T=T)
X_test_seq, y_test_seq = data_classification(test_lob, test_label, T=T)

y_train_all = y_train_seq[:, HORIZON]
y_test = y_test_seq[:, HORIZON]

# 80/20 train/val split
val_split = int(len(X_train_seq) * 0.8)
X_val = X_train_seq[val_split:]
y_val = y_train_all[val_split:]
X_train = X_train_seq[:val_split]
y_train = y_train_all[:val_split]

print(f"Horizon: {HORIZON_NAMES[HORIZON]}")
print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test_seq.shape}")
print(f"\nTest label distribution:")
for i, name in enumerate(['Down', 'Stationary', 'Up']):
    n = (y_test == i).sum()
    print(f"  {name}: {n} ({n/len(y_test)*100:.1f}%)")


Raw train: (149, 254750), Raw test: (149, 139587)
Horizon: k=50
Train: (203720, 100, 40) | Val: (50931, 100, 40) | Test: (139488, 100, 40)

Test label distribution:
  Down: 38408 (27.5%)
  Stationary: 65996 (47.3%)
  Up: 35084 (25.2%)


## 2. Dataset & DataLoaders

In [4]:
class LOBDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

BS = 256
train_ds = LOBDataset(X_train, y_train)
val_ds = LOBDataset(X_val, y_val)
test_ds = LOBDataset(X_test_seq, y_test)

train_ld = DataLoader(train_ds, BS, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
val_ld = DataLoader(val_ds, BS, shuffle=False, num_workers=2, pin_memory=True)
test_ld = DataLoader(test_ds, BS, shuffle=False, num_workers=2, pin_memory=True)

# Compute class weights
counts = torch.bincount(torch.tensor(y_train), minlength=3).float()
class_weights = ((1.0 / counts) / (1.0 / counts).sum() * 3).to(device)
print(f"Input: [{T}, 40], Classes: 3")
print(f"Class weights: {class_weights.cpu().numpy()}")


Input: [100, 40], Classes: 3
Class weights: [0.9467207 1.0937424 0.9595369]


## 3. Base SNN Components (from V1)

In [5]:
class SurrogateSpike(torch.autograd.Function):
    beta = 40.0
    @staticmethod
    def forward(ctx, mem, threshold=1.0):
        ctx.save_for_backward(mem)
        ctx.threshold = threshold
        return (mem >= threshold).float()
    @staticmethod
    def backward(ctx, grad_output):
        mem, = ctx.saved_tensors
        v = mem - ctx.threshold
        grad = 1.0 / (1.0 + SurrogateSpike.beta * torch.abs(v)) ** 2
        return grad_output * grad, None

def spike_fn(x, threshold=1.0):
    return SurrogateSpike.apply(x, threshold)


class ReadoutLayer(nn.Module):
    def __init__(self, input_size, output_size, tau_mem=20.0, dt=10.0):
        super().__init__()
        self.fc = nn.Linear(input_size, output_size, bias=False)
        self.beta = np.exp(-dt / tau_mem)
        nn.init.kaiming_uniform_(self.fc.weight, nonlinearity='linear')
    def forward(self, x):
        B, T_s, _ = x.shape
        mem = torch.zeros(B, self.fc.out_features, device=x.device)
        mem_rec = []
        for t in range(T_s):
            mem = self.beta * mem + (1 - self.beta) * self.fc(x[:, t])
            mem_rec.append(mem)
        return torch.stack(mem_rec, dim=1)


def spike_regularization(all_spikes, theta_l=0.01, s_l=1.0, theta_u=100.0, s_u=0.06):
    reg = torch.tensor(0.0, device=all_spikes[0].device)
    for spk in all_spikes:
        B, T_s, N = spk.shape
        mean_rate = spk.sum(dim=1) / T_s
        reg += s_l / (B * N) * (F.relu(theta_l - mean_rate) ** 2).sum()
        pop_count = spk.sum(dim=(1, 2)) / N
        reg += s_u / B * (F.relu(pop_count - theta_u) ** 2).sum()
    return reg

print("Base components loaded")


Base components loaded


## 4. Trainer (identical to V1)

In [6]:
class Trainer:
    def __init__(self, model, train_ld, val_ld, test_ld, lr=1e-3, device='cuda'):
        self.model = model.to(device)
        self.train_ld, self.val_ld, self.test_ld = train_ld, val_ld, test_ld
        self.device = device
        self.optimizer = torch.optim.Adamax(model.parameters(), lr=lr)
        self.criterion = nn.CrossEntropyLoss(weight=class_weights)
        self.history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [],
                        'spike_rates': [], 'epoch_time': []}

    def _run_epoch(self, loader, train=False):
        self.model.train() if train else self.model.eval()
        total_loss, correct, total, spk_rates = 0, 0, 0, []
        ctx = torch.enable_grad() if train else torch.no_grad()
        with ctx:
            for x, y in loader:
                x, y = x.to(self.device), y.to(self.device)
                logits, all_spk, _ = self.model(x)
                cls_loss = self.criterion(logits, y)
                if train:
                    loss = cls_loss + spike_regularization(all_spk)
                    self.optimizer.zero_grad(); loss.backward()
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                    self.optimizer.step()
                total_loss += cls_loss.item() * x.size(0)
                correct += (logits.argmax(1) == y).sum().item()
                total += x.size(0)
                spk_rates.extend([s.detach().mean().item() for s in all_spk])
        return total_loss/total, correct/total, np.mean(spk_rates)

    def train(self, n_epochs=80, patience=20):
        best_val, best_state, no_improve = 0, None, 0
        for ep in range(n_epochs):
            t0 = time.time()
            tr_l, tr_a, sr = self._run_epoch(self.train_ld, train=True)
            va_l, va_a, _ = self._run_epoch(self.val_ld)
            dt_ep = time.time() - t0
            self.history['train_loss'].append(tr_l); self.history['train_acc'].append(tr_a)
            self.history['val_loss'].append(va_l); self.history['val_acc'].append(va_a)
            self.history['spike_rates'].append(sr); self.history['epoch_time'].append(dt_ep)
            if va_a > best_val + 0.001:
                best_val = va_a; no_improve = 0
                best_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
            else:
                no_improve += 1
            if ep % 5 == 0 or no_improve >= patience:
                star = ' *' if no_improve == 0 else ''
                print(f"Ep {ep:3d} | Tr: {tr_a:.4f} | Va: {va_a:.4f} | Spk: {sr:.4f} | Best: {best_val:.4f} | {dt_ep:.1f}s{star}")
            if no_improve >= patience:
                print(f"Early stop at epoch {ep}"); break
        if best_state:
            self.model.load_state_dict({k: v.to(self.device) for k, v in best_state.items()})
        _, te_a, _ = self._run_epoch(self.test_ld)
        self.history['test_acc'] = te_a
        print(f"\nBest val: {best_val*100:.2f}% | Test: {te_a*100:.2f}% | Time: {sum(self.history['epoch_time'])/60:.1f}min")
        return te_a

print("Trainer ready")


Trainer ready


## 5. Baselines (LSTM + CNN)

In [7]:
class LSTMBaseline(nn.Module):
    def __init__(self, input_size=40, hidden_size=128, n_layers=2, output_size=3, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, n_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size, output_size)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])
    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

class CNNBaseline(nn.Module):
    def __init__(self, input_channels=40, output_size=3):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(input_channels, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64), nn.ReLU(), nn.AdaptiveAvgPool1d(1),
        )
        self.fc = nn.Linear(64, output_size)
    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.conv(x).squeeze(-1)
        return self.fc(x)
    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

def train_baseline(model, train_ld, val_ld, test_ld, n_epochs=60, lr=1e-3):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    best_val, best_state, patience, no_improve = 0, None, 20, 0
    for ep in range(n_epochs):
        model.train()
        for x, y in train_ld:
            x, y = x.to(device), y.to(device)
            loss = criterion(model(x), y)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for x, y in val_ld:
                x, y = x.to(device), y.to(device)
                correct += (model(x).argmax(1) == y).sum().item()
                total += y.size(0)
        va = correct / total
        if va > best_val + 0.001:
            best_val = va; no_improve = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            no_improve += 1
        if ep % 10 == 0: print(f"  Ep {ep}: val={va:.4f} (best={best_val:.4f})")
        if no_improve >= patience:
            print(f"  Early stop at epoch {ep}"); break
    if best_state:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in test_ld:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(1) == y).sum().item()
            total += y.size(0)
    te = correct / total
    print(f"  Best val: {best_val*100:.2f}% | Test: {te*100:.2f}%")
    return te, model

print("--- LSTM ---")
acc_lstm, model_lstm = train_baseline(LSTMBaseline(40, 128, 2, 3, 0.3), train_ld, val_ld, test_ld)

print("\n--- CNN ---")
acc_cnn, model_cnn = train_baseline(CNNBaseline(40, 3), train_ld, val_ld, test_ld)

RESULTS = {'LSTM': acc_lstm, 'CNN': acc_cnn}


--- LSTM ---
  Ep 0: val=0.3700 (best=0.3700)
  Ep 10: val=0.5765 (best=0.5765)
  Ep 20: val=0.5755 (best=0.5868)
  Ep 30: val=0.5743 (best=0.5868)
  Early stop at epoch 35
  Best val: 58.68% | Test: 65.99%

--- CNN ---
  Ep 0: val=0.3700 (best=0.3700)
  Ep 10: val=0.3136 (best=0.4033)
  Ep 20: val=0.3165 (best=0.4142)
  Ep 30: val=0.3136 (best=0.4736)
  Ep 40: val=0.3830 (best=0.4912)
  Ep 50: val=0.3202 (best=0.4912)
  Best val: 53.12% | Test: 65.47%


## 6. V1 Baseline RSNN (threshold=1.0, reproduced)

Exact same architecture as V1. This confirms the baseline on the balanced test set.

In [8]:
class LIFLayer(nn.Module):
    def __init__(self, input_size, hidden_size, recurrent=False,
                 tau_mem_init=20.0, tau_syn_init=10.0, dt=10.0,
                 learnable_tau=False, dropout=0.0, threshold=1.0,
                 learnable_threshold=False):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.recurrent = recurrent
        self.dt = dt
        self.dropout = dropout

        self.W_ff = nn.Linear(input_size, hidden_size, bias=False)
        if recurrent:
            self.W_rec = nn.Linear(hidden_size, hidden_size, bias=False)

        if learnable_tau:
            self.log_tau_mem = nn.Parameter(torch.tensor(np.log(tau_mem_init)))
            self.log_tau_syn = nn.Parameter(torch.tensor(np.log(tau_syn_init)))
        else:
            self.register_buffer('log_tau_mem', torch.tensor(np.log(tau_mem_init)))
            self.register_buffer('log_tau_syn', torch.tensor(np.log(tau_syn_init)))

        # Learnable threshold (LTMD-style)
        if learnable_threshold:
            self.log_threshold = nn.Parameter(torch.tensor(np.log(threshold)))
        else:
            self.register_buffer('log_threshold', torch.tensor(np.log(threshold)))

        nn.init.kaiming_uniform_(self.W_ff.weight, nonlinearity='linear')
        if recurrent:
            nn.init.kaiming_uniform_(self.W_rec.weight, nonlinearity='linear')

    @property
    def alpha(self):
        return torch.exp(-self.dt / torch.exp(self.log_tau_syn))
    @property
    def beta_decay(self):
        return torch.exp(-self.dt / torch.exp(self.log_tau_mem))
    @property
    def threshold(self):
        return torch.exp(self.log_threshold)

    def forward(self, x):
        B, T_steps, _ = x.shape
        alpha, beta = self.alpha, self.beta_decay
        thr = self.threshold
        syn = torch.zeros(B, self.hidden_size, device=x.device)
        mem = torch.zeros(B, self.hidden_size, device=x.device)
        prev_spk = torch.zeros(B, self.hidden_size, device=x.device)
        spk_rec, mem_rec = [], []
        for t in range(T_steps):
            syn = alpha * syn + self.W_ff(x[:, t])
            if self.recurrent:
                rec_in = F.dropout(prev_spk, p=self.dropout, training=self.training) if self.dropout > 0 else prev_spk
                syn = syn + self.W_rec(rec_in)
            mem = beta * mem * (1.0 - prev_spk) + (1.0 - beta) * syn
            spk = spike_fn(mem, thr)
            spk_rec.append(spk); mem_rec.append(mem)
            prev_spk = spk
        return torch.stack(spk_rec, dim=1), torch.stack(mem_rec, dim=1)


class SNN(nn.Module):
    def __init__(self, input_size=40, hidden_size=256, output_size=3,
                 recurrent=True, tau_mem=20.0, tau_syn=10.0, dt=10.0,
                 learnable_tau=True, loss_mode='max_over_time', dropout=0.3,
                 threshold=1.0, learnable_threshold=False):
        super().__init__()
        self.loss_mode = loss_mode
        self.lif = LIFLayer(input_size, hidden_size, recurrent=recurrent,
                            tau_mem_init=tau_mem, tau_syn_init=tau_syn, dt=dt,
                            learnable_tau=learnable_tau, dropout=dropout,
                            threshold=threshold, learnable_threshold=learnable_threshold)
        self.readout = ReadoutLayer(hidden_size, output_size, tau_mem=tau_mem, dt=dt)

    def forward(self, x):
        spikes, _ = self.lif(x)
        out_mem = self.readout(spikes)
        if self.loss_mode == 'max_over_time':
            output, _ = torch.max(out_mem, dim=1)
        else:
            output = out_mem[:, -1, :]
        return output, [spikes], out_mem

    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# Reproduce V1 baseline
print("=" * 60)
print("BASELINE: RSNN Direct (threshold=1.0, V1 reproduction)")
print("=" * 60)
model_base = SNN(40, 256, 3, threshold=1.0, learnable_threshold=False)
print(f"Params: {model_base.count_params():,}")
trainer_base = Trainer(model_base, train_ld, val_ld, test_ld, lr=1e-3, device=device)
acc_base = trainer_base.train(n_epochs=80, patience=20)
RESULTS['RSNN_base'] = acc_base


BASELINE: RSNN Direct (threshold=1.0, V1 reproduction)
Params: 76,546
Ep   0 | Tr: 0.3643 | Va: 0.3700 | Spk: 0.0019 | Best: 0.3700 | 128.6s *
Ep   5 | Tr: 0.3670 | Va: 0.3550 | Spk: 0.0165 | Best: 0.3700 | 127.9s
Ep  10 | Tr: 0.3764 | Va: 0.3356 | Spk: 0.0249 | Best: 0.3700 | 128.7s
Ep  15 | Tr: 0.3810 | Va: 0.3497 | Spk: 0.0349 | Best: 0.3700 | 128.7s
Ep  20 | Tr: 0.3832 | Va: 0.3581 | Spk: 0.0468 | Best: 0.3700 | 127.5s
Early stop at epoch 20

Best val: 37.00% | Test: 37.56% | Time: 44.9min


## 7. Data Diagnosis (confirms root cause)

Measure the actual input magnitudes reaching the LIF layer to calibrate threshold init.

In [9]:
@torch.no_grad()
def diagnose_lif_input(model, loader, device, n_batches=10):
    model.eval()
    all_pre_mem = []
    for i, (x, y) in enumerate(loader):
        if i >= n_batches: break
        x = x.to(device)
        B, T, C = x.shape
        for t in range(T):
            pre = model.lif.W_ff(x[:, t])
            all_pre_mem.append(pre.abs().cpu())
    concat = torch.cat(all_pre_mem, dim=0).flatten()
    # Sample to avoid "tensor too large" error in quantile()
    if len(concat) > 1_000_000:
        idx = torch.randperm(len(concat))[:1_000_000]
        sample = concat[idx]
    else:
        sample = concat
    print("Pre-membrane |W*x| statistics:")
    print(f"  Mean:   {concat.mean():.6f}")
    print(f"  Median: {sample.median():.6f}")
    print(f"  95th:   {sample.quantile(0.95):.6f}")
    print(f"  99th:   {sample.quantile(0.99):.6f}")
    print(f"  Max:    {concat.max():.6f}")
    print(f"  Current threshold: 1.0")
    print(f"  Ratio (95th/threshold): {sample.quantile(0.95)/1.0:.4f}")
    return sample

pre_mem_stats = diagnose_lif_input(model_base, train_ld, device)
p95 = pre_mem_stats.quantile(0.95).item()
print()
print(f"Recommended initial threshold for E1: {p95:.4f}")


Pre-membrane |W*x| statistics:
  Mean:   0.147203
  Median: 0.114706
  95th:   0.411276
  99th:   0.583295
  Max:    0.817075
  Current threshold: 1.0
  Ratio (95th/threshold): 0.4113

Recommended initial threshold for E1: 0.4113


## 8. Experiment E1: Learnable Threshold (LTMD)

**Hypothesis**: If the threshold learns to match the data scale, neurons will fire meaningfully.

Initialise threshold at the 95th percentile of |W*x| from the baseline model.
Use log-parameterisation (threshold = exp(log_threshold)) to keep it positive.

Reference: Wang et al., "LTMD: Learning Improvement of Spiking Neural Networks with Learnable Thresholding Mechanism and Moderate Dropout", NeurIPS 2022.

In [10]:
# Sweep initial thresholds: the diagnosed p95, plus 0.01 and 0.1
init_thresholds = [0.01, 0.1, round(p95, 4)]
init_thresholds = sorted(set(init_thresholds))

print("=" * 60)
print("E1: LEARNABLE THRESHOLD SWEEP")
print("=" * 60)

e1_results = {}
for th_init in init_thresholds:
    print(f"\n--- threshold_init={th_init} ---")
    m = SNN(40, 256, 3, threshold=th_init, learnable_threshold=True)
    print(f"Params: {m.count_params():,} (includes 1 learnable threshold)")
    t = Trainer(m, train_ld, val_ld, test_ld, lr=1e-3, device=device)
    acc = t.train(n_epochs=80, patience=20)
    # Check what threshold learned to
    learned_th = m.lif.threshold.item()
    print(f"  Learned threshold: {learned_th:.6f}")
    e1_results[th_init] = {'acc': acc, 'learned_th': learned_th}
    RESULTS[f'E1_th{th_init}'] = acc

print("\nE1 Summary:")
for th, res in e1_results.items():
    print(f"  init={th:.4f} -> test={res['acc']*100:.2f}%, learned_th={res['learned_th']:.6f}")


E1: LEARNABLE THRESHOLD SWEEP

--- threshold_init=0.01 ---
Params: 76,547 (includes 1 learnable threshold)
Ep   0 | Tr: 0.3587 | Va: 0.3169 | Spk: 0.4260 | Best: 0.3169 | 128.1s *
Ep   5 | Tr: 0.3850 | Va: 0.3700 | Spk: 0.4461 | Best: 0.3700 | 128.5s *
Ep  10 | Tr: 0.3879 | Va: 0.3200 | Spk: 0.4978 | Best: 0.3700 | 129.4s
Ep  15 | Tr: 0.3904 | Va: 0.3353 | Spk: 0.5105 | Best: 0.3700 | 129.2s
Ep  20 | Tr: 0.3944 | Va: 0.3160 | Spk: 0.5306 | Best: 0.3700 | 128.5s
Ep  25 | Tr: 0.3967 | Va: 0.3700 | Spk: 0.5202 | Best: 0.3700 | 128.0s
Early stop at epoch 25

Best val: 37.00% | Test: 41.24% | Time: 55.7min
  Learned threshold: 0.010000

--- threshold_init=0.1 ---
Params: 76,547 (includes 1 learnable threshold)
Ep   0 | Tr: 0.3535 | Va: 0.3695 | Spk: 0.3625 | Best: 0.3695 | 128.0s *
Ep   5 | Tr: 0.3748 | Va: 0.3700 | Spk: 0.4293 | Best: 0.3695 | 128.0s
Ep  10 | Tr: 0.3810 | Va: 0.3700 | Spk: 0.5399 | Best: 0.3695 | 128.7s
Ep  15 | Tr: 0.3861 | Va: 0.3700 | Spk: 0.5993 | Best: 0.3695 | 129.2s

## 9. Experiment E2: Batch Normalisation Through Time (BNTT)

**Hypothesis**: Per-timestep BN rescales inputs to match LIF threshold range without changing the neuron.

Reference: Kim & Panda, "Revisiting Batch Normalization for Training Low-Latency Deep Spiking Neural Networks", Frontiers in Neuroscience 2021.

In [11]:
class LIFLayerBNTT(nn.Module):
    """LIF with per-timestep batch normalisation on the pre-synaptic input."""
    def __init__(self, input_size, hidden_size, n_steps, recurrent=False,
                 tau_mem_init=20.0, tau_syn_init=10.0, dt=10.0,
                 learnable_tau=False, dropout=0.0):
        super().__init__()
        self.hidden_size = hidden_size
        self.recurrent = recurrent
        self.dt = dt
        self.dropout = dropout
        self.n_steps = n_steps

        self.W_ff = nn.Linear(input_size, hidden_size, bias=False)
        if recurrent:
            self.W_rec = nn.Linear(hidden_size, hidden_size, bias=False)

        # BNTT: separate BN params per timestep
        self.bn = nn.ModuleList([nn.BatchNorm1d(hidden_size) for _ in range(n_steps)])

        if learnable_tau:
            self.log_tau_mem = nn.Parameter(torch.tensor(np.log(tau_mem_init)))
            self.log_tau_syn = nn.Parameter(torch.tensor(np.log(tau_syn_init)))
        else:
            self.register_buffer('log_tau_mem', torch.tensor(np.log(tau_mem_init)))
            self.register_buffer('log_tau_syn', torch.tensor(np.log(tau_syn_init)))

        nn.init.kaiming_uniform_(self.W_ff.weight, nonlinearity='linear')
        if recurrent:
            nn.init.kaiming_uniform_(self.W_rec.weight, nonlinearity='linear')

    @property
    def alpha(self):
        return torch.exp(-self.dt / torch.exp(self.log_tau_syn))
    @property
    def beta_decay(self):
        return torch.exp(-self.dt / torch.exp(self.log_tau_mem))

    def forward(self, x):
        B, T_steps, _ = x.shape
        alpha, beta = self.alpha, self.beta_decay
        syn = torch.zeros(B, self.hidden_size, device=x.device)
        mem = torch.zeros(B, self.hidden_size, device=x.device)
        prev_spk = torch.zeros(B, self.hidden_size, device=x.device)
        spk_rec, mem_rec = [], []
        for t in range(T_steps):
            cur = self.W_ff(x[:, t])
            # BNTT: normalise per timestep
            cur = self.bn[t](cur)
            syn = alpha * syn + cur
            if self.recurrent:
                rec_in = F.dropout(prev_spk, p=self.dropout, training=self.training) if self.dropout > 0 else prev_spk
                syn = syn + self.W_rec(rec_in)
            mem = beta * mem * (1.0 - prev_spk) + (1.0 - beta) * syn
            spk = spike_fn(mem)
            spk_rec.append(spk); mem_rec.append(mem)
            prev_spk = spk
        return torch.stack(spk_rec, dim=1), torch.stack(mem_rec, dim=1)


class SNN_BNTT(nn.Module):
    def __init__(self, input_size=40, hidden_size=256, output_size=3, n_steps=100,
                 recurrent=True, tau_mem=20.0, tau_syn=10.0, dt=10.0,
                 learnable_tau=True, dropout=0.3):
        super().__init__()
        self.lif = LIFLayerBNTT(input_size, hidden_size, n_steps, recurrent=recurrent,
                                tau_mem_init=tau_mem, tau_syn_init=tau_syn, dt=dt,
                                learnable_tau=learnable_tau, dropout=dropout)
        self.readout = ReadoutLayer(hidden_size, output_size, tau_mem=tau_mem, dt=dt)

    def forward(self, x):
        spikes, _ = self.lif(x)
        out_mem = self.readout(spikes)
        output, _ = torch.max(out_mem, dim=1)
        return output, [spikes], out_mem

    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


print("=" * 60)
print("E2: BATCH NORMALISATION THROUGH TIME (BNTT)")
print("=" * 60)

model_bntt = SNN_BNTT(40, 256, 3, n_steps=T)
print(f"Params: {model_bntt.count_params():,}")
trainer_bntt = Trainer(model_bntt, train_ld, val_ld, test_ld, lr=1e-3, device=device)
acc_bntt = trainer_bntt.train(n_epochs=80, patience=20)
RESULTS['E2_BNTT'] = acc_bntt


E2: BATCH NORMALISATION THROUGH TIME (BNTT)
Params: 127,746
Ep   0 | Tr: 0.3864 | Va: 0.3281 | Spk: 0.1402 | Best: 0.3281 | 150.9s *
Ep   5 | Tr: 0.4436 | Va: 0.3700 | Spk: 0.1931 | Best: 0.3700 | 149.7s
Ep  10 | Tr: 0.4893 | Va: 0.3700 | Spk: 0.2088 | Best: 0.3700 | 151.6s
Ep  15 | Tr: 0.5224 | Va: 0.3534 | Spk: 0.2113 | Best: 0.3824 | 150.9s
Ep  20 | Tr: 0.5446 | Va: 0.3681 | Spk: 0.2148 | Best: 0.3865 | 151.5s
Ep  25 | Tr: 0.5573 | Va: 0.3715 | Spk: 0.2232 | Best: 0.4087 | 149.7s
Ep  30 | Tr: 0.5685 | Va: 0.4012 | Spk: 0.2268 | Best: 0.4087 | 151.1s
Ep  35 | Tr: 0.5756 | Va: 0.3658 | Spk: 0.2337 | Best: 0.4087 | 148.5s
Ep  40 | Tr: 0.5832 | Va: 0.3680 | Spk: 0.2335 | Best: 0.4087 | 150.1s
Ep  44 | Tr: 0.5901 | Va: 0.3954 | Spk: 0.2352 | Best: 0.4087 | 150.9s
Early stop at epoch 44

Best val: 40.87% | Test: 54.84% | Time: 112.9min


## 10. Experiment E3: Learned Input Gain

**Hypothesis**: A per-feature learnable scalar multiplier before the LIF input lets the network learn the right scale mapping.

This is a lightweight ablation of BNTT — tests whether simple scaling is sufficient vs full normalisation.

In [12]:
class LIFLayerGain(nn.Module):
    """LIF with a learnable per-feature gain before W_ff."""
    def __init__(self, input_size, hidden_size, recurrent=False,
                 tau_mem_init=20.0, tau_syn_init=10.0, dt=10.0,
                 learnable_tau=False, dropout=0.0):
        super().__init__()
        self.hidden_size = hidden_size
        self.recurrent = recurrent
        self.dt = dt
        self.dropout = dropout

        # Learnable per-feature gain (initialised to amplify by ~10x)
        self.gain = nn.Parameter(torch.ones(input_size) * 10.0)

        self.W_ff = nn.Linear(input_size, hidden_size, bias=False)
        if recurrent:
            self.W_rec = nn.Linear(hidden_size, hidden_size, bias=False)

        if learnable_tau:
            self.log_tau_mem = nn.Parameter(torch.tensor(np.log(tau_mem_init)))
            self.log_tau_syn = nn.Parameter(torch.tensor(np.log(tau_syn_init)))
        else:
            self.register_buffer('log_tau_mem', torch.tensor(np.log(tau_mem_init)))
            self.register_buffer('log_tau_syn', torch.tensor(np.log(tau_syn_init)))

        nn.init.kaiming_uniform_(self.W_ff.weight, nonlinearity='linear')
        if recurrent:
            nn.init.kaiming_uniform_(self.W_rec.weight, nonlinearity='linear')

    @property
    def alpha(self):
        return torch.exp(-self.dt / torch.exp(self.log_tau_syn))
    @property
    def beta_decay(self):
        return torch.exp(-self.dt / torch.exp(self.log_tau_mem))

    def forward(self, x):
        B, T_steps, _ = x.shape
        alpha, beta = self.alpha, self.beta_decay
        syn = torch.zeros(B, self.hidden_size, device=x.device)
        mem = torch.zeros(B, self.hidden_size, device=x.device)
        prev_spk = torch.zeros(B, self.hidden_size, device=x.device)
        spk_rec, mem_rec = [], []
        for t in range(T_steps):
            # Apply learnable gain before linear transform
            scaled_input = x[:, t] * self.gain
            syn = alpha * syn + self.W_ff(scaled_input)
            if self.recurrent:
                rec_in = F.dropout(prev_spk, p=self.dropout, training=self.training) if self.dropout > 0 else prev_spk
                syn = syn + self.W_rec(rec_in)
            mem = beta * mem * (1.0 - prev_spk) + (1.0 - beta) * syn
            spk = spike_fn(mem)
            spk_rec.append(spk); mem_rec.append(mem)
            prev_spk = spk
        return torch.stack(spk_rec, dim=1), torch.stack(mem_rec, dim=1)


class SNN_Gain(nn.Module):
    def __init__(self, input_size=40, hidden_size=256, output_size=3,
                 recurrent=True, tau_mem=20.0, tau_syn=10.0, dt=10.0,
                 learnable_tau=True, dropout=0.3):
        super().__init__()
        self.lif = LIFLayerGain(input_size, hidden_size, recurrent=recurrent,
                                tau_mem_init=tau_mem, tau_syn_init=tau_syn, dt=dt,
                                learnable_tau=learnable_tau, dropout=dropout)
        self.readout = ReadoutLayer(hidden_size, output_size, tau_mem=tau_mem, dt=dt)

    def forward(self, x):
        spikes, _ = self.lif(x)
        out_mem = self.readout(spikes)
        output, _ = torch.max(out_mem, dim=1)
        return output, [spikes], out_mem

    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


print("=" * 60)
print("E3: LEARNED INPUT GAIN")
print("=" * 60)

model_gain = SNN_Gain(40, 256, 3)
print(f"Params: {model_gain.count_params():,}")
trainer_gain = Trainer(model_gain, train_ld, val_ld, test_ld, lr=1e-3, device=device)
acc_gain = trainer_gain.train(n_epochs=80, patience=20)
RESULTS['E3_Gain'] = acc_gain


E3: LEARNED INPUT GAIN
Params: 76,586
Ep   0 | Tr: 0.3775 | Va: 0.3700 | Spk: 0.2216 | Best: 0.3700 | 137.7s *
Ep   5 | Tr: 0.3899 | Va: 0.3700 | Spk: 0.2476 | Best: 0.3700 | 136.6s
Ep  10 | Tr: 0.3930 | Va: 0.3700 | Spk: 0.2524 | Best: 0.3700 | 137.2s
Ep  15 | Tr: 0.3955 | Va: 0.3700 | Spk: 0.2750 | Best: 0.3700 | 137.9s
Ep  20 | Tr: 0.3940 | Va: 0.3700 | Spk: 0.3016 | Best: 0.3700 | 136.7s
Early stop at epoch 20

Best val: 37.00% | Test: 42.34% | Time: 48.0min


## 11. Experiment E4: Combined (BNTT + Learnable Threshold)

If E1 and E2 each show improvement, test whether combining them helps further.

In [13]:
class LIFLayerBNTT_LT(nn.Module):
    """BNTT + learnable threshold combined."""
    def __init__(self, input_size, hidden_size, n_steps, recurrent=False,
                 tau_mem_init=20.0, tau_syn_init=10.0, dt=10.0,
                 learnable_tau=False, dropout=0.0, threshold_init=0.1):
        super().__init__()
        self.hidden_size = hidden_size
        self.recurrent = recurrent
        self.dt = dt
        self.dropout = dropout
        self.n_steps = n_steps

        self.W_ff = nn.Linear(input_size, hidden_size, bias=False)
        if recurrent:
            self.W_rec = nn.Linear(hidden_size, hidden_size, bias=False)

        self.bn = nn.ModuleList([nn.BatchNorm1d(hidden_size) for _ in range(n_steps)])
        self.log_threshold = nn.Parameter(torch.tensor(np.log(threshold_init)))

        if learnable_tau:
            self.log_tau_mem = nn.Parameter(torch.tensor(np.log(tau_mem_init)))
            self.log_tau_syn = nn.Parameter(torch.tensor(np.log(tau_syn_init)))
        else:
            self.register_buffer('log_tau_mem', torch.tensor(np.log(tau_mem_init)))
            self.register_buffer('log_tau_syn', torch.tensor(np.log(tau_syn_init)))

        nn.init.kaiming_uniform_(self.W_ff.weight, nonlinearity='linear')
        if recurrent:
            nn.init.kaiming_uniform_(self.W_rec.weight, nonlinearity='linear')

    @property
    def alpha(self):
        return torch.exp(-self.dt / torch.exp(self.log_tau_syn))
    @property
    def beta_decay(self):
        return torch.exp(-self.dt / torch.exp(self.log_tau_mem))
    @property
    def threshold(self):
        return torch.exp(self.log_threshold)

    def forward(self, x):
        B, T_steps, _ = x.shape
        alpha, beta = self.alpha, self.beta_decay
        thr = self.threshold
        syn = torch.zeros(B, self.hidden_size, device=x.device)
        mem = torch.zeros(B, self.hidden_size, device=x.device)
        prev_spk = torch.zeros(B, self.hidden_size, device=x.device)
        spk_rec, mem_rec = [], []
        for t in range(T_steps):
            cur = self.bn[t](self.W_ff(x[:, t]))
            syn = alpha * syn + cur
            if self.recurrent:
                rec_in = F.dropout(prev_spk, p=self.dropout, training=self.training) if self.dropout > 0 else prev_spk
                syn = syn + self.W_rec(rec_in)
            mem = beta * mem * (1.0 - prev_spk) + (1.0 - beta) * syn
            spk = spike_fn(mem, thr)
            spk_rec.append(spk); mem_rec.append(mem)
            prev_spk = spk
        return torch.stack(spk_rec, dim=1), torch.stack(mem_rec, dim=1)


class SNN_Combined(nn.Module):
    def __init__(self, input_size=40, hidden_size=256, output_size=3, n_steps=100,
                 recurrent=True, tau_mem=20.0, tau_syn=10.0, dt=10.0,
                 learnable_tau=True, dropout=0.3, threshold_init=0.1):
        super().__init__()
        self.lif = LIFLayerBNTT_LT(input_size, hidden_size, n_steps, recurrent=recurrent,
                                    tau_mem_init=tau_mem, tau_syn_init=tau_syn, dt=dt,
                                    learnable_tau=learnable_tau, dropout=dropout,
                                    threshold_init=threshold_init)
        self.readout = ReadoutLayer(hidden_size, output_size, tau_mem=tau_mem, dt=dt)

    def forward(self, x):
        spikes, _ = self.lif(x)
        out_mem = self.readout(spikes)
        output, _ = torch.max(out_mem, dim=1)
        return output, [spikes], out_mem

    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


print("=" * 60)
print("E4: BNTT + LEARNABLE THRESHOLD")
print("=" * 60)

model_combo = SNN_Combined(40, 256, 3, n_steps=T, threshold_init=0.1)
print(f"Params: {model_combo.count_params():,}")
trainer_combo = Trainer(model_combo, train_ld, val_ld, test_ld, lr=1e-3, device=device)
acc_combo = trainer_combo.train(n_epochs=80, patience=20)
RESULTS['E4_BNTT_LT'] = acc_combo

# Print learned threshold
print(f"Learned threshold: {model_combo.lif.threshold.item():.6f}")


E4: BNTT + LEARNABLE THRESHOLD
Params: 127,747
Ep   0 | Tr: 0.3858 | Va: 0.3700 | Spk: 0.4335 | Best: 0.3700 | 149.9s *
Ep   5 | Tr: 0.4283 | Va: 0.3700 | Spk: 0.4242 | Best: 0.3700 | 150.1s
Ep  10 | Tr: 0.4899 | Va: 0.3690 | Spk: 0.4683 | Best: 0.3700 | 150.0s
Ep  15 | Tr: 0.5231 | Va: 0.3699 | Spk: 0.5034 | Best: 0.3700 | 150.0s
Ep  20 | Tr: 0.5472 | Va: 0.3693 | Spk: 0.5170 | Best: 0.3864 | 153.0s
Ep  25 | Tr: 0.5657 | Va: 0.3707 | Spk: 0.5373 | Best: 0.3864 | 150.5s
Ep  30 | Tr: 0.5807 | Va: 0.3703 | Spk: 0.5445 | Best: 0.3864 | 150.6s
Ep  35 | Tr: 0.5891 | Va: 0.3896 | Spk: 0.5544 | Best: 0.3896 | 150.9s *
Ep  40 | Tr: 0.5947 | Va: 0.3712 | Spk: 0.5695 | Best: 0.3996 | 149.5s
Ep  45 | Tr: 0.6025 | Va: 0.3905 | Spk: 0.5681 | Best: 0.3996 | 150.8s
Ep  50 | Tr: 0.6065 | Va: 0.4024 | Spk: 0.5678 | Best: 0.4242 | 150.6s
Ep  55 | Tr: 0.6129 | Va: 0.3774 | Spk: 0.5738 | Best: 0.4242 | 151.0s
Ep  60 | Tr: 0.6165 | Va: 0.3928 | Spk: 0.5788 | Best: 0.4242 | 148.9s
Ep  65 | Tr: 0.6189 | Va: 

## 12. Full Evaluation (F1, Confusion Matrix)

In [14]:
@torch.no_grad()
def full_eval(model, loader, device, name):
    model.eval()
    preds, labels = [], []
    for x, y in loader:
        x = x.to(device)
        logits, _, _ = model(x)
        preds.append(logits.argmax(1).cpu())
        labels.append(y)
    preds = torch.cat(preds).numpy()
    labels = torch.cat(labels).numpy()
    acc = (preds == labels).mean()
    f1_w = f1_score(labels, preds, average='weighted')
    f1_m = f1_score(labels, preds, average='macro')
    print(f"\n{name}:")
    print(f"  Accuracy: {acc*100:.2f}%  |  F1 (weighted): {f1_w*100:.2f}%  |  F1 (macro): {f1_m*100:.2f}%")
    print(classification_report(labels, preds, target_names=['Down', 'Stationary', 'Up']))
    return acc, f1_w, f1_m

print("=" * 70)
print("DETAILED EVALUATION ON BALANCED TEST SET (139K samples)")
print("=" * 70)

eval_results = {}
eval_results['RSNN_base'] = full_eval(model_base, test_ld, device, 'RSNN Baseline (th=1.0)')

# Best E1 model
best_e1_th = max(e1_results, key=lambda k: e1_results[k]['acc'])
# Retrain best E1 for eval (or use last trained)
# For simplicity, train fresh
m_e1 = SNN(40, 256, 3, threshold=best_e1_th, learnable_threshold=True)
t_e1 = Trainer(m_e1, train_ld, val_ld, test_ld, lr=1e-3, device=device)
_ = t_e1.train(n_epochs=80, patience=20)
eval_results['E1_LearnTh'] = full_eval(m_e1, test_ld, device, f'E1: Learnable Threshold (init={best_e1_th})')

eval_results['E2_BNTT'] = full_eval(model_bntt, test_ld, device, 'E2: BNTT')
eval_results['E3_Gain'] = full_eval(model_gain, test_ld, device, 'E3: Learned Input Gain')
eval_results['E4_Combined'] = full_eval(model_combo, test_ld, device, 'E4: BNTT + Learnable Threshold')


DETAILED EVALUATION ON BALANCED TEST SET (139K samples)

RSNN Baseline (th=1.0):
  Accuracy: 37.56%  |  F1 (weighted): 32.90%  |  F1 (macro): 28.68%
              precision    recall  f1-score   support

        Down       0.29      0.64      0.39     38408
  Stationary       0.52      0.42      0.47     65996
          Up       0.00      0.00      0.00     35084

    accuracy                           0.38    139488
   macro avg       0.27      0.35      0.29    139488
weighted avg       0.32      0.38      0.33    139488

Ep   0 | Tr: 0.3673 | Va: 0.3685 | Spk: 0.3964 | Best: 0.3685 | 128.9s *
Ep   5 | Tr: 0.3843 | Va: 0.3685 | Spk: 0.4669 | Best: 0.3685 | 128.9s
Ep  10 | Tr: 0.3883 | Va: 0.3485 | Spk: 0.4982 | Best: 0.3700 | 128.4s
Ep  15 | Tr: 0.3902 | Va: 0.3597 | Spk: 0.5385 | Best: 0.3700 | 127.9s
Ep  20 | Tr: 0.3948 | Va: 0.3429 | Spk: 0.5610 | Best: 0.3700 | 129.6s
Ep  25 | Tr: 0.3980 | Va: 0.3639 | Spk: 0.5858 | Best: 0.3700 | 128.9s
Ep  26 | Tr: 0.3988 | Va: 0.3654 | Spk: 0.

## 13. Results Summary

In [15]:
print("=" * 70)
print("COMPLETE RESULTS — FI-2010 LOB (Horizon k=50, Balanced Test Set)")
print("=" * 70)
print(f"{'Model':<45} {'Test Acc':>10} {'vs Base':>10}")
print("-" * 65)

base_acc = RESULTS.get('RSNN_base', 0)
for name, acc in sorted(RESULTS.items(), key=lambda x: x[1], reverse=True):
    delta = (acc - base_acc) * 100
    delta_str = f"{delta:+.1f}pp" if name != 'RSNN_base' else '--'
    print(f"  {name:<43} {acc*100:>8.2f}% {delta_str:>10}")

print("-" * 65)
print(f"  Random baseline: 33.3%")
print()

best_snn = max((v for k, v in RESULTS.items() if k.startswith('E') or k == 'RSNN_base'), default=0)
print(f"Best SNN: {best_snn*100:.2f}%")
print(f"LSTM:     {RESULTS.get('LSTM', 0)*100:.2f}%")
print(f"Gap:      {(RESULTS.get('LSTM', 0) - best_snn)*100:.1f}pp")


COMPLETE RESULTS — FI-2010 LOB (Horizon k=50, Balanced Test Set)
Model                                           Test Acc    vs Base
-----------------------------------------------------------------
  LSTM                                           65.99%    +28.4pp
  CNN                                            65.47%    +27.9pp
  E4_BNTT_LT                                     59.15%    +21.6pp
  E2_BNTT                                        54.84%    +17.3pp
  E3_Gain                                        42.34%     +4.8pp
  E1_th0.01                                      41.24%     +3.7pp
  E1_th0.4113                                    39.84%     +2.3pp
  E1_th0.1                                       37.92%     +0.4pp
  RSNN_base                                      37.56%         --
-----------------------------------------------------------------
  Random baseline: 33.3%

Best SNN: 59.15%
LSTM:     65.99%
Gap:      6.8pp


In [16]:
# Save results
with open('/kaggle/working/fi2010_fix_results.json', 'w') as f:
    json.dump({k: float(v) for k, v in RESULTS.items()}, f, indent=2)
print("Saved to fi2010_fix_results.json")


Saved to fi2010_fix_results.json
